## 🟡 The Gold Layer: Creating Business Views

In the Gold layer, we aggregate and join our cleaned Silver data to create specific "Business Views." For this project, our business requirement is to find the **Top 5 Airlines by Revenue**.

### ➕ Step 1: Create the Gold Data Flow & Add Sources
1. Create a new Data Flow and name it `data serving`.
2. Add your first source: **dim airline**.
   - Instead of a standard dataset, choose **Inline Dataset** and select **Delta**.
   - Point it to your `silver` container and select the `dim_airline` folder.
   - *(Note: With Delta format, always select the folder, not the underlying Parquet files).*
3. Add your second source: **fact bookings** (configured exactly like the airline source, pointing to the `fact_bookings` folder).

### ➕ Step 2: Join Fact and Dimension Tables
We need the airline names (from the dimension table) alongside the ticket costs (from the fact table).
1. Click the **+** icon next to your fact source and select **Join**.
2. **Left stream**: `fact bookings`
3. **Right stream**: `dim airline`
4. **Join type**: Left Outer
5. **Join conditions**: `airline_id` == `airline_id`
6. *(Optional cleanup)*: Use a **Select** activity immediately after the join to drop the redundant `airline_id` column coming from the dimension table to avoid confusion.

### ➕ Step 3: Aggregate the Data (Total Sales)
Now we calculate the total revenue generated by each airline.
1. Add an **Aggregate** activity.
2. **Group by**: Select `airline_name`.
3. **Aggregates**: Create a new column named `total_sales`.
   - Open the expression builder and use the sum function: `sum(ticket_cost)`.

### ➕ Step 4: Window Function (Ranking the Airlines)
To find the top 5, we need to rank them based on total sales.
1. Add a **Window** activity.
2. **Over**: Leave blank (we are ranking the entire dataset, not partitioning it).
3. **Sort**: Select `total_sales` and set the order to **Descending**.
4. **Window columns**: Create a new column named `top`.
   - Open the expression builder and use the function: `dense_rank()`.
   - *(Explanation: `dense_rank()` assigns sequential ranks without skipping numbers if there is a tie, which is an industry standard over the regular `rank()` function).*

### ➕ Step 5: Filter for Top 5
1. Add a **Filter** activity.
2. **Condition**: `top <= 5`. This ensures only the top 5 performing airlines pass through.

### ➕ Step 6: Sink to the Gold Layer
Because this is a business view that we want to completely refresh every time the pipeline runs, we will use *Overwrite* instead of Upsert.
1. Add an **Alter Row** activity and set the condition to **Insert if**: `1 == 1`.
2. Add a **Sink** activity.
3. **Dataset**: Inline Dataset -> Delta.
4. **Settings**: Browse to your new `gold` container and name the folder `business_view`.
5. **Settings Tab (Crucial)**: Check the **Allow insert** box (ensure Upsert is unchecked) and set the table action to **Clear the folder** or **Overwrite** so it completely refreshes the top 5 list each time.
